# Experiment 003.1: Query2Doc + Dense (SILMA Kashif-2B)

**Model:** SILMA Kashif-2B-Instruct-v1.0

**Enhancement:** Query2Doc (LLM-based pseudo-document generation)

**Retrieval:** Dense (mDPR)

**Baseline:** Experiment 001 (NDCG@10 = 0.4993)

**Reference:** Experiment 003 (Qwen 2.5 3B, NDCG@10 = 0.5435)

---

## Model Details

- **Size:** 2B parameters (smallest in comparison)
- **Purpose:** Built for Arabic RAG (extractive QA)
- **Architecture:** Based on Gemma
- **Quantization:** None (FP16)
- **Expected VRAM:** ~4-5 GB
- **Expected Runtime:** ~15-20 minutes (fastest model)

---

## Setup

In [ ]:
# Install dependencies
!apt-get install -qq openjdk-21-jdk-headless
!pip install -q pyserini faiss-cpu datasets accelerate bitsandbytes transformers torch

Selecting previously unselected package openjdk-21-jre-headless:amd64.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../openjdk-21-jre-headless_21.0.10+7-1~22.04_amd64.deb ...
Unpacking openjdk-21-jre-headless:amd64 (21.0.10+7-1~22.04) ...
Selecting previously unselected package openjdk-21-jdk-headless:amd64.
Preparing to unpack .../openjdk-21-jdk-headless_21.0.10+7-1~22.04_amd64.deb ...
Unpacking openjdk-21-jdk-headless:amd64 (21.0.10+7-1~22.04) ...
Setting up openjdk-21-jre-headless:amd64 (21.0.10+7-1~22.04) ...
update-alternatives: using /usr/lib/jvm/java-21-openjdk-amd64/bin/java to provide /usr/bin/java (java) in auto mode
update-alternatives: using /usr/lib/jvm/java-21-openjdk-amd64/bin/jpackage to provide /usr/bin/jpackage (jpackage) in auto mode
update-alternatives: using /usr/lib/jvm/java-21-openjdk-amd64/bin/keytool to provide /usr/bin/keytool (keytool) in auto mode
update-alternatives: using /usr/lib/jvm/java-21-openjdk-amd64/b

In [ ]:
# Restart runtime after installation
# Runtime -> Restart runtime

In [ ]:
import os
import sys

# Set Java home
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'

# Clone repo if not exists
if not os.path.exists('/content/graduation'):
    !git clone https://github.com/Osmanoor/graduation.git

# Add to path
sys.path.insert(0, '/content/graduation/arabic-rag-query-enhancement')

print("Setup complete!")

Cloning into 'graduation'...
remote: Enumerating objects: 541, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 541 (delta 24), reused 67 (delta 22), pack-reused 468 (from 1)
Receiving objects: 100% (541/541), 20.52 MiB | 9.78 MiB/s, done.
Resolving deltas: 100% (196/196), done.
Setup complete!


## Load Data

In [ ]:
from src.utils.data_loader import MIRACLDataLoader

# Load MIRACL Arabic dev set
loader = MIRACLDataLoader(language="ar", split="dev")
topics, qrels = loader.load_all()

print(f"Loaded {len(topics)} queries")
print(f"Loaded {len(qrels)} qrels")

Loading topics from miracl-v1.0-ar-dev...
✓ Loaded 2896 queries
Loading qrels from miracl-v1.0-ar-dev...
✓ Loaded qrels for 2896 queries
Loaded 2896 queries
Loaded 2896 qrels


## Initialize SILMA Kashif-2B Query2Doc Enhancer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.enhancers.query2doc import Query2DocEnhancer

# Model configuration
MODEL_NAME = "silma-ai/SILMA-Kashif-2B-Instruct-v1.0"
MAX_NEW_TOKENS = 128
TEMPERATURE = 0.1
TOP_P = 0.9
BATCH_SIZE = 16  # Larger batch for small model

print(f"Loading {MODEL_NAME}...")
print(f"Expected VRAM: ~4-5 GB")

# Create enhancer
enhancer = Query2DocEnhancer(
    model_name=MODEL_NAME,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    batch_size=BATCH_SIZE
)

print("\nSILMA Kashif-2B loaded successfully!")
print(f"Model device: {enhancer.model.device}")
print(f"Batch size: {BATCH_SIZE}")

Loading silma-ai/SILMA-Kashif-2B-Instruct-v1.0...
Expected VRAM: ~4-5 GB
Loading silma-ai/SILMA-Kashif-2B-Instruct-v1.0 in float16...


config.json:   0%|          | 0.00/974 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

✓ Model loaded on cuda:0
✓ Batch size: 16 (processing 16 queries at once)
✓ Max tokens: 128 (shorter = faster)

SILMA Kashif-2B loaded successfully!
Model device: cuda:0
Batch size: 16


## Check GPU Status

In [ ]:
if torch.cuda.is_available():
    print("=== GPU Status ===")
    print(f"CUDA available: True")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"GPU memory total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("CUDA not available - using CPU (will be slow)")

=== GPU Status ===
CUDA available: True
GPU name: Tesla T4
GPU memory allocated: 4.87 GB
GPU memory total: 14.56 GB


In [ ]:
# SILMA doesn't support system role, so we need to modify the enhance methods
# to combine system prompt with user message

def enhance_silma(self, query: str, query_id: str = None) -> str:
    """Enhanced method for SILMA (no system role)"""
    # Combine system prompt with query in user message
    combined_message = f"{self.system_prompt}\n\nQuery: {query}"

    messages = [
        {"role": "user", "content": combined_message}
    ]

    text = self.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

    with torch.no_grad():
        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=self.max_new_tokens,
            temperature=self.temperature,
            top_p=self.top_p,
            do_sample=True,
            pad_token_id=self.tokenizer.pad_token_id
        )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    pseudo_doc = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    enhanced = f"{query} {pseudo_doc}"
    return enhanced

def enhance_batch_parallel_silma(self, queries, query_ids=None):
    """Enhanced batch method for SILMA (no system role)"""
    # Combine system prompt with each query
    all_messages = [
        [{"role": "user", "content": f"{self.system_prompt}\n\nQuery: {query}"}]
        for query in queries
    ]

    texts = [
        self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        for messages in all_messages
    ]

    model_inputs = self.tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(self.model.device)

    with torch.no_grad():
        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=self.max_new_tokens,
            temperature=self.temperature,
            top_p=self.top_p,
            do_sample=True,
            pad_token_id=self.tokenizer.pad_token_id
        )

    input_lengths = model_inputs.input_ids.shape[1]
    pseudo_docs = self.tokenizer.batch_decode(
        generated_ids[:, input_lengths:],
        skip_special_tokens=True
    )

    enhanced = [f"{q} {doc}" for q, doc in zip(queries, pseudo_docs)]
    return enhanced

# Monkey-patch the enhancer methods
import types
enhancer.enhance = types.MethodType(enhance_silma, enhancer)
enhancer.enhance_batch_parallel = types.MethodType(enhance_batch_parallel_silma, enhancer)

print("SILMA-specific methods applied (no system role)")

SILMA-specific methods applied (no system role)


## Test Enhancement (First 5 Queries)

In [ ]:
# Test on first 5 queries
query_ids = list(topics.keys())[:5]
test_queries = [topics[qid]['title'] for qid in query_ids]

print("Testing SILMA Kashif-2B on first 5 queries...\n")

for i, (qid, query) in enumerate(zip(query_ids, test_queries), 1):
    enhanced = enhancer.enhance(query, qid)
    print(f"Query {i} ({qid}):")
    print(f"Original: {query}")
    print(f"Enhanced: {enhanced[:200]}...")
    print(f"Length: {len(query)} -> {len(enhanced)} chars")
    print("-" * 80)

print("\nTest complete. Check if outputs are in Arabic and relevant.")

Testing SILMA Kashif-2B on first 5 queries...

Query 1 (8099):
Original: من هو علي بن محمد السمري؟
Enhanced: من هو علي بن محمد السمري؟ علي بن محمد السمري هو أحد الرواة الذين عاشوا في القرن الثاني الهجري. كان له كتاب 'الآية المختص' الذي يعتبر من أهم الكتب في الحديث....
Length: 25 -> 156 chars
--------------------------------------------------------------------------------
Query 2 (3640):
Original: متى تم إستخدام الغوّاصات لأول مرة؟
Enhanced: متى تم إستخدام الغوّاصات لأول مرة؟ تم استخدام الغوّاصات لأول مرة في القرن التاسع عشر في القرن التاسع عشر....
Length: 34 -> 105 chars
--------------------------------------------------------------------------------
Query 3 (4971):
Original: من هو القديس المسمى بالصخرة؟
Enhanced: من هو القديس المسمى بالصخرة؟ القديس المسمى بالصخرة هو القديس بطرس بن مالك بن أبي بكر، وهو من الملائكة الذين سُجّل في الملائكة في المئوية....
Length: 28 -> 137 chars
--------------------------------------------------------------------------------
Query 4 (1461):
Original: هل ي

## Enhance All Queries

In [ ]:
import time

# Prepare all queries
query_ids = list(topics.keys())
queries = [topics[qid]['title'] for qid in query_ids]

print(f"Enhancing {len(queries)} queries with SILMA Kashif-2B...")
print(f"Batch size: {BATCH_SIZE}")
print(f"Expected time: ~15-20 minutes\n")

start_time = time.time()

# Enhance all queries
enhanced_queries = enhancer.enhance_batch(queries)

elapsed = time.time() - start_time
print(f"\nEnhancement complete!")
print(f"Total time: {elapsed/60:.1f} minutes")
print(f"Queries per minute: {len(queries)/(elapsed/60):.1f}")

Enhancing 2896 queries with SILMA Kashif-2B...
Batch size: 16
Expected time: ~15-20 minutes




Enhancing batches: 100%|██████████| 181/181 [17:23<00:00,  5.76s/it]


Enhancement complete!
Total time: 17.4 minutes
Queries per minute: 166.6


## Save Enhanced Queries

In [ ]:
import pickle

# Save enhanced queries
output_file = "silma_2b_temp01.pkl"

with open(output_file, 'wb') as f:
    pickle.dump({
        'query_ids': query_ids,
        'original_queries': queries,
        'enhanced_queries': enhanced_queries,
        'model': MODEL_NAME,
        'config': {
            'max_new_tokens': MAX_NEW_TOKENS,
            'temperature': TEMPERATURE,
            'top_p': TOP_P,
            'batch_size': BATCH_SIZE
        }
    }, f)

print(f"Enhanced queries saved to: {output_file}")
print(f"File size: {os.path.getsize(output_file) / 1024**2:.1f} MB")

Enhanced queries saved to: silma_2b_temp01.pkl
File size: 0.8 MB


## Query Expansion Statistics

In [ ]:
import numpy as np

# Calculate statistics
original_lengths = [len(q) for q in queries]
enhanced_lengths = [len(eq) for eq in enhanced_queries]
expansion_ratios = [e/o if o > 0 else 0 for o, e in zip(original_lengths, enhanced_lengths)]

print("=== Query Expansion Statistics ===")
print(f"\nOriginal queries:")
print(f"  Mean length: {np.mean(original_lengths):.1f} chars")
print(f"  Median length: {np.median(original_lengths):.1f} chars")

print(f"\nEnhanced queries:")
print(f"  Mean length: {np.mean(enhanced_lengths):.1f} chars")
print(f"  Median length: {np.median(enhanced_lengths):.1f} chars")

print(f"\nExpansion ratio:")
print(f"  Mean: {np.mean(expansion_ratios):.2f}x")
print(f"  Median: {np.median(expansion_ratios):.2f}x")

print(f"\nNote: SILMA is extractive-focused, may produce shorter expansions than generative models.")

=== Query Expansion Statistics ===

Original queries:
  Mean length: 29.5 chars
  Median length: 27.0 chars

Enhanced queries:
  Mean length: 116.6 chars
  Median length: 93.0 chars

Expansion ratio:
  Mean: 4.51x
  Median: 3.13x

Note: SILMA is extractive-focused, may produce shorter expansions than generative models.


## Initialize Dense Retriever (mDPR)

In [ ]:
from src.retrievers.dense import mDPRRetriever
import gc

# Clear GPU memory
del enhancer
torch.cuda.empty_cache()
gc.collect()

print("Initializing mDPR retriever...")
retriever = mDPRRetriever()

print("mDPR retriever ready")

## Retrieve with Enhanced Queries

In [ ]:
print(f"Retrieving with {len(enhanced_queries)} enhanced queries...")
print("This will take ~2-3 minutes\n")

start_time = time.time()

# Search
results = retriever.search(enhanced_queries, k=100)

elapsed = time.time() - start_time
print(f"\nRetrieval complete in {elapsed:.1f} seconds")

## Evaluate

In [ ]:
from src.evaluation.metrics import RetrievalEvaluator, print_metrics

# Evaluate
evaluator = RetrievalEvaluator(qrels)
metrics = evaluator.evaluate(results)

print("\n" + "="*60)
print("EXPERIMENT 003.1: Query2Doc + Dense (SILMA Kashif-2B)")
print("="*60)
print_metrics(metrics)

# Comparison with baselines
print("\n" + "="*60)
print("COMPARISON")
print("="*60)

baseline_dense = {
    'ndcg_cut_10': 0.4993,
    'recall_10': 0.6156,
    'recall_100': 0.8407,
    'recip_rank': 0.5328
}

qwen_25_3b = {
    'ndcg_cut_10': 0.5435,
    'recall_10': 0.6608,
    'recall_100': 0.8594,
    'recip_rank': 0.5742
}

print("\nBaseline (Exp 001 - Identity):")
print(f"  NDCG@10:    {baseline_dense['ndcg_cut_10']:.4f}")
print(f"  Recall@10:  {baseline_dense['recall_10']:.4f}")
print(f"  Recall@100: {baseline_dense['recall_100']:.4f}")
print(f"  MRR:        {baseline_dense['recip_rank']:.4f}")

print("\nQwen 2.5 3B (Exp 003 - Reference):")
print(f"  NDCG@10:    {qwen_25_3b['ndcg_cut_10']:.4f}")
print(f"  Recall@10:  {qwen_25_3b['recall_10']:.4f}")
print(f"  Recall@100: {qwen_25_3b['recall_100']:.4f}")
print(f"  MRR:        {qwen_25_3b['recip_rank']:.4f}")

print("\nSILMA Kashif-2B (This experiment):")
print(f"  NDCG@10:    {metrics['ndcg_cut_10']:.4f} ({(metrics['ndcg_cut_10']/baseline_dense['ndcg_cut_10']-1)*100:+.2f}% vs baseline)")
print(f"  Recall@10:  {metrics['recall_10']:.4f} ({(metrics['recall_10']/baseline_dense['recall_10']-1)*100:+.2f}% vs baseline)")
print(f"  Recall@100: {metrics['recall_100']:.4f} ({(metrics['recall_100']/baseline_dense['recall_100']-1)*100:+.2f}% vs baseline)")
print(f"  MRR:        {metrics['recip_rank']:.4f} ({(metrics['recip_rank']/baseline_dense['recip_rank']-1)*100:+.2f}% vs baseline)")

print("\nComparison with Qwen 2.5 3B:")
print(f"  NDCG@10:    {(metrics['ndcg_cut_10']/qwen_25_3b['ndcg_cut_10']-1)*100:+.2f}%")
print(f"  Recall@10:  {(metrics['recall_10']/qwen_25_3b['recall_10']-1)*100:+.2f}%")
print(f"  Recall@100: {(metrics['recall_100']/qwen_25_3b['recall_100']-1)*100:+.2f}%")
print(f"  MRR:        {(metrics['recip_rank']/qwen_25_3b['recip_rank']-1)*100:+.2f}%")

## Save Results

In [ ]:
import json

# Create results directory
os.makedirs('results/silma_kashif_2b_dense', exist_ok=True)

# Save metrics
with open('results/silma_kashif_2b_dense/exp_003_1_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("Results saved to: results/silma_kashif_2b_dense/exp_003_1_metrics.json")

## Summary

**Model:** SILMA Kashif-2B-Instruct-v1.0

**Size:** 2B parameters (smallest model tested)

**Purpose:** Arabic RAG (extractive QA)

**Quantization:** None (FP16)

**Key Observations:**
- Fastest model (~15-20 min for enhancement)
- Smallest VRAM footprint (~4-5 GB)
- Extractive-focused (may produce shorter expansions)
- Built specifically for Arabic RAG tasks

**Next Steps:**
1. Compare with other models in Task 4.0b
2. If performance is good, test on BM25S
3. Document findings in model comparison table